In [15]:
import linecache
import json
import requests
import os
from datetime import datetime
import pandas as pd

In [16]:
current_path = os.getcwd()
print("Current Working Directory:", current_path)

Current Working Directory: /home/achum/de_course/goal_1/portfolio/Project1_Document_Streaming/code/client


In [17]:
line = linecache.getline('../../output/output.txt', 1)
# print('line')
# print(line)
# write the line to the API
myjson = json.loads(line)

# print('')
# print('myjson')
print(myjson)

{'trans_num': '0b242abb623afc578575680df30655b9', 'trans_date_trans_time': '2019-01-01 00:00:18', 'cc_num': 2703186189652095, 'merchant': 'fraud_Rippin, Kub and Mann', 'category': 'misc_net', 'amt': 4.97, 'card_holder': {'first': 'Jennifer', 'last': 'Banks', 'gender': 'F', 'dob': '1988-03-09'}, 'job': 'Psychologist, counselling', 'address': {'street': '561 Perry Cove', 'city': 'Moravian Falls', 'state': 'NC', 'zip': 28654, 'lat': 36.0788, 'long': -81.1781}, 'city_pop': 3495, 'merch_location': {'lat': 36.011293, 'long': -82.048315}, 'unix_time': 1325376018, 'is_fraud': 0}


In [23]:
print(myjson['trans_date_trans_time'])
print(datetime.strptime(myjson['trans_date_trans_time'],'%Y-%m-%d %H:%M:%S').strftime("%-d/%-m/%Y %H:%M:%S"))

2019-01-01 00:00:18
1/1/2019 00:00:18


In [32]:
# Transform 'dob' from string to datetime
# myjson['card_holder']['dob'] = 
datetime.strptime(myjson['card_holder']['dob'], '%Y-%m-%d').strftime("%-d/%-m/%Y")

'9/3/1988'

# Testing fastapi

In [25]:
from pydantic import BaseModel
from datetime import datetime
import json

# Define the models using Pydantic
class CardHolder(BaseModel):
    first: str
    last: str
    gender: str
    dob: str  # Change to string for easier parsing

class Address(BaseModel):
    street: str
    city: str
    state: str
    zip: int
    lat: float
    long: float

class MerchLocation(BaseModel):
    lat: float
    long: float

class CreditTrx(BaseModel):
    trans_num: str
    trans_date_trans_time: str  # Change to string for easier parsing
    cc_num: int
    merchant: str
    category: str
    amt: float
    card_holder: CardHolder
    job: str
    address: Address
    city_pop: int
    merch_location: MerchLocation
    unix_time: int
    is_fraud: int

# Sample JSON data for testing
data = {
    "trans_num": "8a6293af5ed278dea14448ded2685fea",
    "trans_date_trans_time": "2019-01-01 00:09:03",
    "cc_num": 3514865930894695,
    "merchant": "fraud_Beier-Hyatt",
    "category": "shopping_pos",
    "amt": 7.77,
    "card_holder": {
        "first": "Christopher",
        "last": "Castaneda",
        "gender": "M",
        "dob": "1967-08-30"
    },
    "job": "Naval architect",
    "address": {
        "street": "1632 Cohen Drive Suite 639",
        "city": "High Rolls Mountain Park",
        "state": "NM",
        "zip": 88325,
        "lat": 32.9396,
        "long": -105.8189
    },
    "city_pop": 899,
    "merch_location": {
        "lat": 32.863258,
        "long": -106.520205
    },
    "unix_time": 1325376543,
    "is_fraud": 0
}

# Function to process the transaction
def process_transaction(data):
    try:
        # Create an instance of CreditTrx using the data
        item = CreditTrx(**data)

        # Validate and parse datetime data (trx and dob)
        trx_date = datetime.strptime(item.trans_date_trans_time, '%Y-%m-%d %H:%M:%S').strftime("%-d/%-m/%Y %H:%M:%S")
        item.trans_date_trans_time = trx_date
        print('Parsed transaction timestamp: ', item.trans_date_trans_time)

        dob_date = datetime.strptime(item.card_holder.dob, '%Y-%m-%d').strftime("%-d/%-m/%Y")
        item.card_holder.dob = dob_date
        print('Parsed date of birth: ', item.card_holder.dob)

        # Convert the item back to JSON
        json_of_item = item.json()
        print("JSON representation of the item:", json_of_item)

    except ValueError as e:
        print("ValueError:", e)


In [26]:
data = myjson

In [27]:
# Run the function with sample data
process_transaction(data)

Parsed transaction timestamp:  1/1/2019 00:00:18
Parsed date of birth:  9/3/1988
JSON representation of the item: {"trans_num":"0b242abb623afc578575680df30655b9","trans_date_trans_time":"1/1/2019 00:00:18","cc_num":2703186189652095,"merchant":"fraud_Rippin, Kub and Mann","category":"misc_net","amt":4.97,"card_holder":{"first":"Jennifer","last":"Banks","gender":"F","dob":"9/3/1988"},"job":"Psychologist, counselling","address":{"street":"561 Perry Cove","city":"Moravian Falls","state":"NC","zip":28654,"lat":36.0788,"long":-81.1781},"city_pop":3495,"merch_location":{"lat":36.011293,"long":-82.048315},"unix_time":1325376018,"is_fraud":0}
